# 10-bob. 10-BOB. ANOMALIYALARNI ANIQLASH
## Amaliy misol: uch xil detektor va kontekstning ahamiyati

[![Colab'da ochish](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mrmsi/si-kiberxavfsizlik/blob/main/10-bob/10-1_uch-xil-detektor-va-kontekstning.ipynb)

*"Sun'iy intellektni kiberxavfsizlikda qo'llash" darsligi, 10-bob.*

Quyidagi dasturda serverning har soatlik ko'rsatkichlari sun'iy ravishda yaratiladi: soat, chiquvchi trafik hajmi va tizimga kirishlar soni. Kunduzi (soat 9 dan 18 gacha) trafik va kirishlar ko'p, tunda esa kam. Ikki xil anomaliya qo'shiladi: nuqtaviy anomaliya (kunduzi odatdagidan besh barobar katta trafik) va kontekstual anomaliya (kunduzgi faollikning aynan o'zi, lekin tungi soat 3 da). Uchta detektor ikki holatda sinab ko'riladi: soat haqidagi ma'lumot berilmaganda va berilganda. Soat aylana bo'ylab kodlanadi, shunda model soat 23 va soat 0 bir-biriga yaqin ekanini tushunadi. Dastur amaliy mashg'ulotning A10 mavzusiga mos keladi.

**Ishga tushirish:** yuqoridagi menyudan *Runtime → Run all* (*Ishga tushirish → Hammasini bajarish*) ni tanlang yoki kataklarni birma-bir Shift+Enter bilan bajaring. Barcha ma'lumotlar dasturning o'zida yaratiladi, qo'shimcha fayl yuklash shart emas.

In [1]:
import numpy as np
from sklearn.ensemble import IsolationForest
from sklearn.svm import OneClassSVM
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler

rng = np.random.default_rng(1)

def kuzatuvlar(n, soat=None, trafik=None):
    """Serverning har soatlik ko'rsatkichlari: soat, chiquvchi trafik (MB), kirishlar soni."""
    s = rng.integers(0, 24, n) if soat is None else np.full(n, soat)
    kunduzi = (s >= 9) & (s <= 18)
    t = np.where(kunduzi, rng.normal(500, 80, n), rng.normal(60, 20, n)) if trafik is None \
        else rng.normal(trafik, 30, n)
    k = np.where(kunduzi, rng.poisson(40, n), rng.poisson(3, n))
    return np.column_stack([s, t, k])

oddiy = kuzatuvlar(5000)
nuqtaviy = kuzatuvlar(50, soat=14, trafik=2500)          # kunduzi juda katta trafik
kontekstual = kuzatuvlar(50, soat=14)                     # kunduzgi faoliyat...
kontekstual[:, 0] = 3                                     # ...lekin tungi soat 3 da
sinov_oddiy = kuzatuvlar(2000)

def xususiyatlar(X, kontekst):
    t, k = X[:, 1:2], X[:, 2:3]
    if not kontekst:
        return np.hstack([t, k])
    burchak = 2 * np.pi * X[:, 0:1] / 24                  # soatni aylana bo'ylab kodlash
    return np.hstack([t, k, np.sin(burchak), np.cos(burchak)])

for kontekst in [False, True]:
    olch = StandardScaler().fit(xususiyatlar(oddiy, kontekst))
    f = lambda X: olch.transform(xususiyatlar(X, kontekst))
    A = f(oddiy)

    # Avtokodlovchi: 2 neyronli "tor joy" orqali ma'lumotni qayta tiklaydi
    ak = MLPRegressor(hidden_layer_sizes=(8, 2, 8), max_iter=3000, random_state=0).fit(A, A)
    xato = lambda X: ((ak.predict(f(X)) - f(X)) ** 2).mean(axis=1)
    chegara = np.quantile(xato(oddiy), 0.99)

    detektorlar = {
        "Izolyatsiya o'rmoni": IsolationForest(contamination=0.01, random_state=0).fit(A),
        "Bir sinfli SVM": OneClassSVM(nu=0.01, gamma="scale").fit(A),
    }
    print("Soat xususiyati bilan:" if kontekst else "Soat xususiyatisiz:")
    for nom, d in detektorlar.items():
        topildi = lambda X: np.mean(d.predict(f(X)) == -1)
        print(f"  {nom:<20} nuqtaviy {topildi(nuqtaviy):>4.0%}  kontekstual {topildi(kontekstual):>4.0%}"
              f"  noto'g'ri signal {topildi(sinov_oddiy):.1%}")
    print(f"  {'Avtokodlovchi':<20} nuqtaviy {np.mean(xato(nuqtaviy) > chegara):>4.0%}"
          f"  kontekstual {np.mean(xato(kontekstual) > chegara):>4.0%}"
          f"  noto'g'ri signal {np.mean(xato(sinov_oddiy) > chegara):.1%}")

Soat xususiyatisiz:
  Izolyatsiya o'rmoni  nuqtaviy  92%  kontekstual   2%  noto'g'ri signal 1.1%
  Bir sinfli SVM       nuqtaviy 100%  kontekstual   0%  noto'g'ri signal 1.1%
  Avtokodlovchi        nuqtaviy 100%  kontekstual   2%  noto'g'ri signal 1.2%


Soat xususiyati bilan:
  Izolyatsiya o'rmoni  nuqtaviy  14%  kontekstual  68%  noto'g'ri signal 1.0%
  Bir sinfli SVM       nuqtaviy 100%  kontekstual 100%  noto'g'ri signal 1.3%
  Avtokodlovchi        nuqtaviy 100%  kontekstual  98%  noto'g'ri signal 1.1%


## Kitobdagi natija

Dasturni ishga tushirganingizda quyidagiga o'xshash natija chiqishi kerak (tasodifiy sonlar generatori urug'i qat'iy belgilangani uchun natija odatda aynan bir xil bo'ladi; faqat bajarilish vaqti va har safar yangi yaratiladigan tasodifiy kalitlar farq qilishi mumkin):

```
Soat xususiyatisiz:
  Izolyatsiya o'rmoni  nuqtaviy  92%  kontekstual   2%  noto'g'ri signal 1.1%
  Bir sinfli SVM       nuqtaviy 100%  kontekstual   0%  noto'g'ri signal 1.1%
  Avtokodlovchi        nuqtaviy 100%  kontekstual   2%  noto'g'ri signal 1.2%
Soat xususiyati bilan:
  Izolyatsiya o'rmoni  nuqtaviy  14%  kontekstual  68%  noto'g'ri signal 1.0%
  Bir sinfli SVM       nuqtaviy 100%  kontekstual 100%  noto'g'ri signal 1.3%
  Avtokodlovchi        nuqtaviy 100%  kontekstual  98%  noto'g'ri signal 1.1%
```

## Natijalar tahlili
 Natijalardan uchta muhim xulosa kelib chiqadi.

*Birinchidan, kontekst berilmasa, kontekstual anomaliyalar ko'rinmaydi.* Soat haqida ma'lumot bo'lmaganda, uchala detektor ham tungi faollikning deyarli hammasini o'tkazib yubordi (0–2 foiz). Bu tushunarli: modelga faqat trafik va kirishlar soni berilgan, ular esa oddiy kunduzgi qiymatlardan farq qilmaydi. Soat qo'shilgandan keyin bir sinfli SVM va avtokodlovchi tungi faollikni deyarli to'liq aniqladi. Demak, anomaliyalarni aniqlashda algoritm tanlashdan oldin "qaysi kontekstda g'ayrioddiy?" degan savolga javob berish va kerakli kontekst belgilarini qo'shish muhim.

*Ikkinchidan, nuqtaviy anomaliyalarni barcha usullar yaxshi aniqlaydi, lekin istisnolar bor.* Soat qo'shilmaganda besh barobar katta trafikni barcha detektorlar osongina topdi. Biroq soat qo'shilgandan keyin izolyatsiya o'rmonining natijasi keskin yomonlashdi: endi u bunday kuzatuvlarning atigi 14 foizini topdi. Buning sababi algoritmning ishlash tamoyilida. Izolyatsiya o'rmoni bo'linish uchun belgini tasodifiy tanlaydi va endi bo'linishlarning yarmi soat belgilari bo'yicha amalga oshiriladi, bu kuzatuvlar esa soat bo'yicha g'ayrioddiy emas. Bundan tashqari, o'qitish ma'lumotlaridagi eng katta qiymatdan ham kattaroq bo'lgan kuzatuvlar daraxtda o'sha eng katta qiymatli kuzatuv bilan bir joyga tushadi. Shuning uchun trafik besh emas, yuz barobar katta bo'lganda ham ularning bali deyarli o'zgarmaydi. Bu misol hech bir algoritm hamma holatlar uchun eng yaxshi emasligini va bir nechta usulni sinab ko'rish kerakligini ko'rsatadi.

*Uchinchidan, noto'g'ri signallar ulushi taxminan bir xil.* Barcha detektorlarda u 1–1,3 foiz atrofida, chunki biz chegarani o'qitish ma'lumotlarining taxminan 1 foizi anomaliya deb belgilanadigan qilib tanladik. Bu chegara aniqlash sifati bilan noto'g'ri signallar o'rtasidagi kelishuvdir.

## O'zingiz sinab ko'ring

Parametrlarni (ma'lumotlar hajmi, chegara qiymati, tasodifiy son urug'i) o'zgartirib, natija qanday o'zgarishini kuzating. Kitobdagi 10-bobning B darajadagi topshiriqlari aynan shu daftar asosida bajariladi.